# Face Matcher Unit Tests

These tests execute the actual `match_face_embedding` function from `face.py` without importing the Streamlit app, so they do not start the UI, access a webcam, or initialize the face detector. The test vectors are synthetic and do not require model downloads or image files.

In [1]:
import ast
import unittest
from pathlib import Path

import numpy as np

source_file = Path.cwd() / "face.py"
if not source_file.is_file():
    raise FileNotFoundError(f"Run this notebook from the project directory; not found: {source_file}")

syntax_tree = ast.parse(source_file.read_text(encoding="utf-8"))
matcher_node = next(
    node for node in syntax_tree.body
    if isinstance(node, ast.FunctionDef) and node.name == "match_face_embedding"
)
matcher_module = ast.Module(body=[matcher_node], type_ignores=[])
matcher_namespace = {"np": np}
exec(compile(matcher_module, str(source_file), "exec"), matcher_namespace)
match_face_embedding = matcher_namespace["match_face_embedding"]

In [2]:
class MatchFaceEmbeddingTests(unittest.TestCase):
    def setUp(self):
        self.cache = {
            "Alice": {
                "embeddings": np.array([[1.0, 0.0], [0.8, 0.6]], dtype=np.float32),
                "status": "Authorised",
            },
            "Bob": {
                "embeddings": np.array([[0.0, 1.0]], dtype=np.float32),
                "status": "Blacklisted",
            },
        }

    def test_returns_closest_identity_and_status(self):
        name, status, distance, similarity = match_face_embedding(
            np.array([0.99, 0.01], dtype=np.float32),
            self.cache,
            threshold=1.13,
        )

        self.assertEqual(name, "Alice")
        self.assertEqual(status, "Authorised")
        self.assertLess(distance, 0.03)
        self.assertGreater(similarity, 98.0)

    def test_rejects_match_beyond_threshold(self):
        far_cache = {
            "Alice": {
                "embeddings": np.array([[0.0, 1.0]], dtype=np.float32),
                "status": "Authorised",
            }
        }

        name, status, distance, similarity = match_face_embedding(
            np.array([1.0, 0.0], dtype=np.float32),
            far_cache,
            threshold=1.13,
        )

        self.assertIsNone(name)
        self.assertIsNone(status)
        self.assertAlmostEqual(distance, np.sqrt(2.0), places=5)
        self.assertEqual(similarity, 0.0)

    def test_empty_query_or_cache_returns_no_match(self):
        no_query = match_face_embedding(None, self.cache)
        no_cache = match_face_embedding(np.array([1.0, 0.0]), {})

        for result in (no_query, no_cache):
            self.assertIsNone(result[0])
            self.assertIsNone(result[1])
            self.assertTrue(np.isinf(result[2]))
            self.assertEqual(result[3], 0.0)

In [3]:
test_suite = unittest.defaultTestLoader.loadTestsFromTestCase(MatchFaceEmbeddingTests)
unittest.TextTestRunner(verbosity=2).run(test_suite)

test_empty_query_or_cache_returns_no_match (__main__.MatchFaceEmbeddingTests.test_empty_query_or_cache_returns_no_match) ... ok
test_rejects_match_beyond_threshold (__main__.MatchFaceEmbeddingTests.test_rejects_match_beyond_threshold) ... ok
test_returns_closest_identity_and_status (__main__.MatchFaceEmbeddingTests.test_returns_closest_identity_and_status) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.011s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

In [ ]:
def run_photo_comparison():
    import cv2

    query_image = Path.cwd() / "ite-pass.jpg"
    people_dir = Path.cwd() / "people"
    registered_images = sorted(
        path for path in people_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    if not query_image.is_file():
        raise FileNotFoundError(f"Query image not found: {query_image}")
    if not registered_images:
        raise FileNotFoundError(f"No registered images found in: {people_dir}")

    deepface_import = next(
        node for node in syntax_tree.body
        if isinstance(node, ast.ImportFrom)
        and node.module == "deepface"
        and any(alias.name == "DeepFace" for alias in node.names)
    )
    embedding_node = next(
        node for node in syntax_tree.body
        if isinstance(node, ast.FunctionDef) and node.name == "extract_face_embedding"
    )
    config_names = {"MODEL_NAME", "DETECTOR_BACKEND", "MATCH_THRESHOLD"}
    config_nodes = [
        node for node in syntax_tree.body
        if isinstance(node, ast.Assign)
        and any(
            isinstance(target, ast.Name) and target.id in config_names
            for target in node.targets
        )
    ]
    face_helpers = ast.Module(
        body=[*config_nodes, deepface_import, embedding_node],
        type_ignores=[],
    )
    exec(compile(face_helpers, str(source_file), "exec"), matcher_namespace)
    extract_face_embedding = matcher_namespace["extract_face_embedding"]
    model_name = matcher_namespace["MODEL_NAME"]
    detector_backend = matcher_namespace["DETECTOR_BACKEND"]
    threshold = matcher_namespace["MATCH_THRESHOLD"]

    face_cache = {}
    for image_path in registered_images:
        image = cv2.imread(str(image_path))
        embedding = extract_face_embedding(image)
        if embedding is None:
            print(f"Skipping {image_path.name}: could not extract a face embedding")
            continue
        face_cache[image_path.stem] = {
            "embeddings": np.asarray([embedding], dtype=np.float32),
            "status": "Blacklisted",
        }

    query_pixels = cv2.imread(str(query_image))
    query_embedding = extract_face_embedding(query_pixels)
    if query_embedding is None:
        raise ValueError(f"Could not extract a face embedding from {query_image.name}")
    if not face_cache:
        raise ValueError("No registered image produced a face embedding")

    ranked_matches = []
    for name, person_data in face_cache.items():
        distance = float(np.min(np.linalg.norm(
            person_data["embeddings"] - query_embedding,
            axis=1,
        )))
        similarity = max(0.0, min(100.0, (1.0 - distance / 2.0) * 100.0))
        ranked_matches.append((distance, name, person_data["status"], similarity))

    ranked_matches.sort()
    print(f"Query image: {query_image.name}")
    print(
        f"Model: {model_name}; detector: {detector_backend}; "
        f"Euclidean L2 threshold: {threshold:.2f}\n"
    )
    for distance, name, status, similarity in ranked_matches:
        match_result = "MATCH" if distance <= threshold else "NO MATCH"
        print(
            f"{name}: {match_result} | status={status} | "
            f"distance={distance:.4f} | similarity={similarity:.1f}%"
        )

    best_name, best_status, best_distance, best_similarity = match_face_embedding(
        query_embedding,
        face_cache,
        threshold=threshold,
    )
    if best_name is None:
        print(f"\nOverall result: Unknown (nearest distance={best_distance:.4f})")
    else:
        print(
            f"\nOverall result: {best_name} ({best_status}) | "
            f"distance={best_distance:.4f} | similarity={best_similarity:.1f}%"
        )


try:
    run_photo_comparison()
except Exception as error:
    print(f"Photo comparison could not run: {type(error).__name__}: {error}")

26-09-26 02:45:13 - ⚠️ 
 ⚠️ DEPRECATION WARNING:
 Running 'pip install deepface' alone will no longer be sufficient and will
 NOT install a default backend in an upcoming major release.

 Currently, TensorFlow is included by default, but this behavior will be deprecated.
 Please explicitly specify your preferred backend engine when installing:

   -> pip install deepface[tensorflow]
   -> pip install deepface[pytorch]

 Otherwise, you will encounter 'module not found' errors.

Photo comparison could not run: AttributeError: module 'tensorflow._api.v2.compat.v2.__internal__' has no attribute 'register_load_context_function'
